# MELA-TPU (JAX) on a Colab TPU
Runtime -> Change runtime type -> TPU. Upload `MELA-TPU.zip` when asked. Steps: unzip, check devices, run the CPU-vs-TPU equivalence smoke, time one training step at d=256 B=16 T=512 (2 layers).

In [ ]:
import jax, jax.numpy as jnp, os, sys, time
print(jax.__version__, jax.devices())
from google.colab import files
if not os.path.exists('MELA-TPU'):
    up = files.upload()
    import zipfile; zipfile.ZipFile(list(up.keys())[0]).extractall('.')
sys.path.insert(0, 'MELA-TPU')
from melatpu import core, model

In [ ]:
# 1) small equivalence: TPU vs CPU backend on the same inputs (fp32 matmuls forced)
pass  # precision is scoped inside core._mm (HIGHEST on the transport and chain only); no global flag
cfg = core.config(64, 256, chunk=64, recompute=False)
key = jax.random.PRNGKey(0)
P = core.init_params(key, cfg)
h = jax.random.normal(jax.random.PRNGKey(1), (2, 256, 64), jnp.float32)
n_ev = len(range(cfg['k_event'], 256, cfg['k_event']))
us = [jax.random.uniform(jax.random.PRNGKey(10 + i), (2, cfg['n_walks'], cfg['walk_len'] + 1, 2)) for i in range(n_ev)]
f = jax.jit(lambda P, h, us: core.layer_forward(P, cfg, h, us))
o_tpu, inst = f(P, h, us)
cpu = jax.devices('cpu')[0]
with jax.default_device(cpu):
    o_cpu, _ = jax.jit(lambda P, h, us: core.layer_forward(P, cfg, h, us))(jax.device_put(P, cpu), jax.device_put(h, cpu), jax.device_put(us, cpu))
print('TPU vs CPU rel', float(jnp.abs(jax.device_get(o_tpu) - jax.device_get(o_cpu)).max() / jnp.abs(jax.device_get(o_cpu)).max()))
print({k: float(v) for k, v in inst[0].items()})

In [ ]:
# 2) training step at the frozen sizing (d 256, B 16, T 512, 2 layers) on every visible TPU chip,
#    plus the compiled peak memory -- the survey's open question (fused transport+chain scan, j0.5)
d, T, V, layers = 256, 512, 65, 2
B = 16 * max(1, jax.device_count() // 8)      # batch axis is sharded: keep B a multiple of the chip count
cfg = core.config(d, T)
key = jax.random.key(0)
P = model.init_lm(key, V, cfg, layers)
mesh = model.data_mesh()
opt, step = model.make_train_step(cfg, mesh=mesh)
st = opt.init(P)
x = jax.random.randint(jax.random.key(2), (B, T), 0, V)
lowered = jax.jit(lambda P, st, x, y, k: step(P, st, x, y, k)).lower(P, st, x, x, key).compile()
mem = lowered.memory_analysis()
print('devices', jax.device_count(), jax.devices()[0].device_kind,
      '| peak temp %.2f GB' % (mem.temp_size_in_bytes / 1e9),
      '| args %.2f GB' % (mem.argument_size_in_bytes / 1e9))
for i in range(2):                                   # compile + warm
    P, st, loss, insts = step(P, st, x, x, jax.random.fold_in(key, i)); loss.block_until_ready()
ts = []
for i in range(3):
    t0 = time.perf_counter(); P, st, loss, insts = step(P, st, x, x, jax.random.fold_in(key, 10 + i)); loss.block_until_ready()
    ts.append(time.perf_counter() - t0)
print('JAX TPU step s (median of 3):', round(sorted(ts)[1], 4), '| loss', float(loss))
print({k: round(float(v), 6) for k, v in insts[0][0].items()})


In [ ]:
# 3) THE decisive lever: transport precision. HIGHEST = 6 bf16 passes (the fp32 contract),
#    HIGH = 3, DEFAULT = 1. Cost is ~70% of the layer, so this sets the TPU budget.
#    Gate: orth_drift < 1e-3 (the frozen design's criterion) at the cheapest passing setting.
import importlib
for name in ('HIGHEST', 'HIGH', 'DEFAULT'):
    core._HI = getattr(jax.lax.Precision, name)          # scoped to expm + squarings + chain only
    opt2, step2 = model.make_train_step(cfg, mesh=mesh)
    P2 = model.init_lm(key, V, cfg, layers); st2 = opt2.init(P2)
    for i in range(2):
        P2, st2, loss2, ins2 = step2(P2, st2, x, x, jax.random.fold_in(key, i)); loss2.block_until_ready()
    ts = []
    for i in range(3):
        t0 = time.perf_counter(); P2, st2, loss2, ins2 = step2(P2, st2, x, x, jax.random.fold_in(key, 30 + i))
        loss2.block_until_ready(); ts.append(time.perf_counter() - t0)
    e = ins2[0][0]
    print('%-8s step %.4f s | orth_drift %.2e | hol_norm %.4f | loss %.4f'
          % (name, sorted(ts)[1], float(e['orth_drift']), float(e['hol_norm']), float(loss2)))
core._HI = jax.lax.Precision.HIGHEST                     # restore the contract


In [ ]:
# 4) width sweep: does the MXU alignment rule (n = d/4 -> 64 / 128 / 256) show up in step time,
#    and does the memory fit one chip? Stops at the first width that OOMs.
for d_ in (256, 512, 1024):
    try:
        c = core.config(d_, 512); k = jax.random.key(7)
        Pw = model.init_lm(k, V, c, 2); ow, sw = model.make_train_step(c, mesh=mesh); stw = ow.init(Pw)
        Bw = 2 * jax.device_count()
        xb = jax.random.randint(jax.random.key(8), (Bw, 512), 0, V)
        m = jax.jit(lambda P, s, a, b, kk: sw(P, s, a, b, kk)).lower(Pw, stw, xb, xb, k).compile().memory_analysis()
        for i in range(2): Pw, stw, l, ins = sw(Pw, stw, xb, xb, jax.random.fold_in(k, i)); l.block_until_ready()
        t0 = time.perf_counter(); Pw, stw, l, ins = sw(Pw, stw, xb, xb, jax.random.fold_in(k, 9)); l.block_until_ready()
        print('d %4d (n %3d, M %4d) B%d: %.3f s/step | peak temp %.2f GB | orth_drift %.1e'
              % (d_, c['n'], c['M'], Bw, time.perf_counter() - t0, m.temp_size_in_bytes / 1e9, float(ins[0][0]['orth_drift'])))
    except Exception as ex:
        print('d', d_, 'FAILED:', str(ex)[:160]); break
